# DeBERTa-v3-base PCL Detection (Trainer API + Prompt-Based Classification)

This notebook uses the Hugging Face Trainer with a **prompt-based (cloze-style) classification** approach following the BEIKE-NLP paper:

1. **Prompt template** (BEIKE-NLP Section 3.1): each paragraph is wrapped as `"{text} Is it patronizing or condescending? [MASK]"`, converting classification into a masked language modeling task.
2. **Multi-word verbalizer**: each class maps to multiple label words (e.g., yes/okay/true/surely for PCL). The model averages the MLM-head probabilities of all label words per class.
3. **Weighted random sampler** for class imbalance.
4. **Trainer API** for clean training/evaluation loops.

**Verbalizer:**
- Class 1 (PCL):    yes, okay, true, surely
- Class 0 (No PCL): no, false, never, neither


In [1]:
!pip install contractions huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.8 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login

login(token=hf_token)


In [3]:
import os
import re
import random
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import WeightedRandomSampler

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns
import contractions

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cpu


In [4]:
# ============================================================
# Configuration
# ============================================================
MODEL_NAME = 'microsoft/deberta-v3-large'
MAX_LENGTH = 256
NUM_LABELS = 2

NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 400
SAMPLER_POWER = 1.0  # 1.0=1/k (balanced), 0.5=1/sqrt(k)

EVAL_EVERY = 20
LOG_EVERY = 50

# Multi-word verbalizer (BEIKE-NLP style)
VERBALIZER = {
    0: ['no', 'false', 'never', 'neither'],   # No-PCL
    1: ['yes', 'okay', 'true', 'surely'],     # PCL
}

# Prompt template (BEIKE-NLP Section 3.1)
# {text} is the preprocessed paragraph, {mask} is replaced with [MASK]
PROMPT_TEMPLATE = '{text} Is it patronizing or condescending? {mask}'

if os.path.exists('/kaggle/input'):
    DATA_ROOT = '/kaggle/input/datasets/wowthecoder/patronizing-and-condescending-language-detection'
else:
    DATA_ROOT = '.'

TSV_PATH = os.path.join(DATA_ROOT, 'dontpatronizeme_pcl.tsv')
TRAIN_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'train_semeval_parids-labels.csv')
DEV_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'dev_semeval_parids-labels.csv')

OUTPUT_DIR = 'checkpoints/deberta_v3_prompt_trainer'
LOG_DIR = 'logs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'DATA_ROOT   : {DATA_ROOT}')
print(f'TSV_PATH    : {TSV_PATH}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR}')


DATA_ROOT   : /kaggle/input/datasets/wowthecoder/patronizing-and-condescending-language-detection
TSV_PATH    : /kaggle/input/datasets/wowthecoder/patronizing-and-condescending-language-detection/dontpatronizeme_pcl.tsv
OUTPUT_DIR  : checkpoints/deberta_v3_prompt_trainer


In [5]:
def load_task1(train_path: str) -> pd.DataFrame:
    """
    Load Task 1 data and convert original labels to binary:
      0/1 -> 0 (No-PCL)
      2/3/4 -> 1 (PCL)
    """
    rows = []
    with open(train_path, encoding='utf-8') as f:
        for line in f.readlines()[4:]:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 6:
                continue

            par_id = parts[0]
            art_id = parts[1]
            keyword = parts[2]
            country = parts[3]
            text = parts[4]
            orig_label = parts[-1]
            label = 0 if orig_label in {'0', '1'} else 1

            rows.append(
                {
                    'par_id': str(par_id),
                    'art_id': art_id,
                    'keyword': keyword,
                    'country': country,
                    'text': text,
                    'label': label,
                    'orig_label': orig_label,
                }
            )

    return pd.DataFrame(
        rows,
        columns=['par_id', 'art_id', 'keyword', 'country', 'text', 'label', 'orig_label'],
    )


def preprocess_text(text: str) -> str:
    text = str(text)
    text = contractions.fix(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = load_task1(TSV_PATH)
df['clean_text'] = df['text'].apply(preprocess_text)

print(f'Loaded dataset: {len(df):,} samples')
print(df['label'].value_counts().sort_index().rename({0: 'No-PCL', 1: 'PCL'}))


Loaded dataset: 10,469 samples
label
No-PCL    9476
PCL        993
Name: count, dtype: int64


In [6]:
# ============================================================
# Official Train/Dev split
# ============================================================
train_ids_df = pd.read_csv(TRAIN_IDS_PATH, dtype={'par_id': str})
dev_ids_df = pd.read_csv(DEV_IDS_PATH, dtype={'par_id': str})

train_par_ids = set(train_ids_df['par_id'].astype(str))
dev_par_ids = set(dev_ids_df['par_id'].astype(str))

train_df = df[df['par_id'].isin(train_par_ids)].copy().reset_index(drop=True)
dev_df = df[df['par_id'].isin(dev_par_ids)].copy().reset_index(drop=True)

leftover_df = df[~df['par_id'].isin(train_par_ids | dev_par_ids)].copy().reset_index(drop=True)
if len(leftover_df) > 0:
    train_df = pd.concat([train_df, leftover_df], ignore_index=True)
    print(f'Appended {len(leftover_df):,} unassigned samples to training set.')


def describe_split(name: str, frame: pd.DataFrame):
    n = len(frame)
    n_pcl = int((frame['label'] == 1).sum())
    n_no_pcl = int((frame['label'] == 0).sum())
    ratio = f'1:{(n_no_pcl / n_pcl):.1f}' if n_pcl > 0 else 'undefined'
    print(f'{name:<6} -> total={n:,} | PCL={n_pcl:,} | No-PCL={n_no_pcl:,} | ratio={ratio}')


describe_split('Train', train_df)
describe_split('Dev', dev_df)


Train  -> total=8,375 | PCL=794 | No-PCL=7,581 | ratio=1:9.5
Dev    -> total=2,094 | PCL=199 | No-PCL=1,895 | ratio=1:9.5


In [7]:
# ============================================================
# Tokenization with prompt wrapping + Hugging Face Datasets
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MASK_TOKEN = tokenizer.mask_token
MASK_TOKEN_ID = tokenizer.mask_token_id
print(f'Mask token: {MASK_TOKEN!r}  (id={MASK_TOKEN_ID})')

# -- Resolve multi-word verbalizer token IDs --
def resolve_verbalizer_ids(tokenizer, verbalizer):
    class_ids = {}
    for label_idx in sorted(verbalizer.keys()):
        ids = []
        for word in verbalizer[label_idx]:
            tokens = tokenizer.encode(' ' + word, add_special_tokens=False)
            tok_id = tokens[0]
            n = len(tokens)
            tag = 'single' if n == 1 else f'multi({n})'
            print(f'  Class {label_idx} -> "{word}" -> id {tok_id}  [{tag}]')
            ids.append(tok_id)
        class_ids[label_idx] = ids
    return class_ids

print('\nVerbalizer token IDs:')
VERBALIZER_IDS = resolve_verbalizer_ids(tokenizer, VERBALIZER)
print(f'Resolved: {VERBALIZER_IDS}')

# -- Build prompted text --
def build_prompt(text, mask_token):
    return PROMPT_TEMPLATE.format(text=text, mask=mask_token)

# Smoke test
demo = build_prompt('These poor children really need our help', MASK_TOKEN)
print(f'\nExample prompt:\n{demo}')
print(f'Tokens: {len(tokenizer(demo)["input_ids"])}')

# -- Create HF datasets with prompted text --
def add_prompt_column(df):
    df = df.copy()
    df['prompted_text'] = df['clean_text'].apply(lambda t: build_prompt(t, MASK_TOKEN))
    return df

train_prompted = add_prompt_column(train_df)
dev_prompted = add_prompt_column(dev_df)

train_hf = Dataset.from_pandas(
    train_prompted[['prompted_text', 'label']].rename(columns={'prompted_text': 'text'}),
    preserve_index=False,
)
dev_hf = Dataset.from_pandas(
    dev_prompted[['prompted_text', 'label']].rename(columns={'prompted_text': 'text'}),
    preserve_index=False,
)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = train_hf.map(tokenize, batched=True, remove_columns=['text'])
dev_ds = dev_hf.map(tokenize, batched=True, remove_columns=['text'])

train_ds = train_ds.rename_column('label', 'labels')
dev_ds = dev_ds.rename_column('label', 'labels')

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
dev_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Weighted random sampler weights
n_train = len(train_df)
n_pcl = int((train_df['label'] == 1).sum())
n_no_pcl = int((train_df['label'] == 0).sum())
k_p = n_pcl / n_train
k_n = n_no_pcl / n_train

w_pcl = 1.0 / (k_p ** SAMPLER_POWER)
w_no_pcl = 1.0 / (k_n ** SAMPLER_POWER)
sample_weights = np.where(train_df['label'].values == 1, w_pcl, w_no_pcl).astype(np.float64)
expected_pos_rate = (n_pcl * w_pcl) / ((n_pcl * w_pcl) + (n_no_pcl * w_no_pcl))

print(f'\nSAMPLER_POWER         : {SAMPLER_POWER:.2f}')
print(f'Sampling weights      : PCL={w_pcl:.4f} | No-PCL={w_no_pcl:.4f}')
print(f'Expected sampled PCL% : {expected_pos_rate*100:.2f}%')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f'\nTrain: {train_ds}')
print(f'Dev:   {dev_ds}')


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Mask token: '[MASK]'  (id=128000)

Verbalizer token IDs:
  Class 0 -> "no" -> id 363  [single]
  Class 0 -> "false" -> id 3770  [single]
  Class 0 -> "never" -> id 518  [single]
  Class 0 -> "neither" -> id 3899  [single]
  Class 1 -> "yes" -> id 2489  [single]
  Class 1 -> "okay" -> id 4385  [single]
  Class 1 -> "true" -> id 980  [single]
  Class 1 -> "surely" -> id 4740  [single]
Resolved: {0: [363, 3770, 518, 3899], 1: [2489, 4385, 980, 4740]}

Example prompt:
These poor children really need our help Is it patronizing or condescending? [MASK]
Tokens: 15


Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]


SAMPLER_POWER         : 1.00
Sampling weights      : PCL=10.5479 | No-PCL=1.1047
Expected sampled PCL% : 50.00%

Train: Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8375
})
Dev:   Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2094
})


In [8]:
# ============================================================
# Metrics (handles log-prob outputs from prompt model)
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_no_pcl': f1_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'f1_pcl': f1_score(labels, preds, pos_label=1, average='binary', zero_division=0),
        'prec_no_pcl': precision_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'prec_pcl': precision_score(labels, preds, pos_label=1, average='binary', zero_division=0),
        'rec_no_pcl': recall_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'rec_pcl': recall_score(labels, preds, pos_label=1, average='binary', zero_division=0),
    }


In [11]:
# ============================================================
# Model: Prompt-based DeBERTa with multi-word verbalizer
# ============================================================
from transformers import AutoModelForMaskedLM, AutoConfig
from transformers.modeling_outputs import SequenceClassifierOutput
import torch.nn as nn
import torch.nn.functional as F


class PromptMLMClassifier(nn.Module):
    """
    Wraps AutoModelForMaskedLM for prompt-based classification (BEIKE-NLP style).

    Forward pass:
    1. Run input through DeBERTa MLM head -> vocab logits at every position
    2. Find the [MASK] position in each sample
    3. Softmax over full vocabulary at [MASK] position
    4. Average probabilities of label words per class (multi-word verbalizer)
    5. Return log-probs and NLL loss (compatible with Trainer)

    This class returns a SequenceClassifierOutput so that Trainer can
    extract .loss and .logits automatically.
    """

    def __init__(self, model_name, verbalizer_ids, mask_token_id):
        super().__init__()
        self.mlm = AutoModelForMaskedLM.from_pretrained(model_name)
        self.verbalizer_ids = verbalizer_ids  # {0: [id,...], 1: [id,...]}
        self.mask_token_id = mask_token_id
        self.num_classes = len(verbalizer_ids)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.mlm(input_ids=input_ids, attention_mask=attention_mask)
        vocab_logits = outputs.logits  # (batch, seq_len, vocab_size)

        # Find [MASK] position in each sample
        batch_size = input_ids.shape[0]
        # mask_positions: for each sample, get index of first [MASK] token
        mask_flags = (input_ids == self.mask_token_id)
        # Use argmax to get first True position per row; if no mask found, falls back to 0
        mask_pos = mask_flags.long().argmax(dim=-1)  # (batch,)

        batch_idx = torch.arange(batch_size, device=input_ids.device)
        mask_logits = vocab_logits[batch_idx, mask_pos, :]  # (batch, vocab_size)

        # Softmax over full vocab, then average label-word probs per class
        mask_probs = torch.softmax(mask_logits, dim=-1)  # (batch, vocab_size)

        class_probs = []
        for cls_idx in sorted(self.verbalizer_ids.keys()):
            word_ids = torch.tensor(
                self.verbalizer_ids[cls_idx],
                device=input_ids.device, dtype=torch.long,
            )
            cls_prob = mask_probs[:, word_ids].mean(dim=-1)  # (batch,)
            class_probs.append(cls_prob)

        class_probs = torch.stack(class_probs, dim=-1)  # (batch, num_classes)
        log_probs = torch.log(class_probs + 1e-10)

        loss = None
        if labels is not None:
            loss = F.nll_loss(log_probs, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=log_probs,
        )


model = PromptMLMClassifier(
    model_name=MODEL_NAME,
    verbalizer_ids=VERBALIZER_IDS,
    mask_token_id=MASK_TOKEN_ID,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

# ============================================================
# TrainingArguments + WeightedSamplerTrainer
# ============================================================
args_kwargs = {
    'output_dir': OUTPUT_DIR,
    'num_train_epochs': NUM_EPOCHS,
    'per_device_train_batch_size': TRAIN_BATCH_SIZE,
    'per_device_eval_batch_size': EVAL_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'warmup_steps': WARMUP_STEPS,
    'lr_scheduler_type': 'cosine',
    'logging_steps': LOG_EVERY,
    'save_strategy': 'steps',
    'save_steps': EVAL_EVERY,
    'eval_steps': EVAL_EVERY,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1_pcl',
    'greater_is_better': True,
    'save_total_limit': 2,
    'report_to': 'none',
    'seed': SEED,
    'fp16': False,
    'bf16': torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
}

# Handle transformers API change: evaluation_strategy -> eval_strategy
ta_sig = inspect.signature(TrainingArguments.__init__).parameters
if 'eval_strategy' in ta_sig:
    args_kwargs['eval_strategy'] = 'steps'
else:
    args_kwargs['evaluation_strategy'] = 'steps'

training_args = TrainingArguments(**args_kwargs)


class WeightedSamplerTrainer(Trainer):
    def __init__(self, *args, sample_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.sample_weights = sample_weights

    def _get_train_sampler(self, dataset=None):
        if self.sample_weights is None:
            return super()._get_train_sampler(dataset)
        if self.train_dataset is None:
            return None
        return WeightedRandomSampler(
            weights=torch.as_tensor(self.sample_weights, dtype=torch.double),
            num_samples=len(self.sample_weights),
            replacement=True,
        )

    def _save(self, output_dir=None, state_dict=None):
        """
        Save the inner MLM model (PreTrainedModel) instead of the wrapper.
        This avoids the safetensors shared-memory error caused by tied weights
        (embedding <-> decoder weight, and cls.predictions.bias <-> decoder.bias).
        """
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)

        # Unwrap DataParallel / DDP if needed
        unwrapped = self.model
        if hasattr(unwrapped, 'module'):
            unwrapped = unwrapped.module

        # The inner .mlm is a PreTrainedModel — its save_pretrained
        # correctly handles tied weights
        unwrapped.mlm.save_pretrained(output_dir)

        # Save tokenizer
        if self.processing_class is not None:
            self.processing_class.save_pretrained(output_dir)

        # Save training args
        torch.save(self.args, os.path.join(output_dir, 'training_args.bin'))

        # Save verbalizer config for reproducibility
        import json as _json
        vc = {
            'verbalizer': {str(k): v for k, v in VERBALIZER.items()},
            'verbalizer_ids': {str(k): v for k, v in VERBALIZER_IDS.items()},
            'prompt_template': PROMPT_TEMPLATE,
            'mask_token_id': unwrapped.mask_token_id,
        }
        with open(os.path.join(output_dir, 'verbalizer_config.json'), 'w') as f:
            _json.dump(vc, f, indent=2)


trainer = WeightedSamplerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    sample_weights=sample_weights,
)

print('Trainer configured.')


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
DebertaV2ForMaskedLM LOAD REPORT from: microsoft/deberta-v3-base
Key                                           | Status     | 
----------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias         | UNEXPECTED | 
deberta.embeddings.position_embeddings.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias                   | UNEXPECTED | 
cls.predictions.transform.dense.bias          | MISSING    | 
cls.predictions.transform.LayerNorm.bias      | MISSING    | 
cls.predictions.transform.LayerNorm.weight    | MISSING    | 
cls.

Trainable parameters: 184,551,780
Trainer configured.


In [ ]:
train_result = trainer.train()
train_result


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
eval_metrics = trainer.evaluate()
print('Evaluation metrics:')
for k, v in eval_metrics.items():
    if isinstance(v, (int, float)):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')


In [ ]:
# ============================================================
# Detailed evaluation report
# ============================================================
pred_output = trainer.predict(dev_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)

print(classification_report(y_true, y_pred, target_names=['No-PCL', 'PCL'], digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Pred No-PCL', 'Pred PCL'],
    yticklabels=['True No-PCL', 'True PCL'],
)
plt.title('Confusion Matrix (Dev)')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Save best model
# ============================================================
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, 'best')
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# Save the inner MLM model and tokenizer
model.mlm.save_pretrained(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

# Also save verbalizer config for reproducibility
import json
verbalizer_config = {
    'verbalizer': {str(k): v for k, v in VERBALIZER.items()},
    'verbalizer_ids': {str(k): v for k, v in VERBALIZER_IDS.items()},
    'prompt_template': PROMPT_TEMPLATE,
    'mask_token_id': MASK_TOKEN_ID,
}
with open(os.path.join(BEST_MODEL_DIR, 'verbalizer_config.json'), 'w') as f:
    json.dump(verbalizer_config, f, indent=2)

print(f'Saved best model, tokenizer, and verbalizer config to: {BEST_MODEL_DIR}')
